# Neuro — oil-spill segmentation U-Net (Colab trainer)

Self-contained. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
~1–2 h. Produces `oilspill_unet.pt` — download it, drop it in `neuro/backend/weights/`,
and set in `neuro/backend/.env`:
```
SEGMENTATION_MODEL=trained-unet
SEGMENTATION_WEIGHTS=weights/oilspill_unet.pt
```

Data: Sentinel-1 SAR Oil Spill Dataset Part I — Zenodo 10.5281/zenodo.8346860 (CC-BY-4.0).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime type to GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
!pip -q install tifffile

In [ ]:
# --- download + extract the dataset (images 40.7 GB, masks 6 MB) ---
import os, subprocess, pathlib
os.makedirs('data', exist_ok=True)
B = 'https://zenodo.org/api/records/8346860/files'
for name in ['01_Train_Val_Oil_Spill_images.7z', '01_Train_Val_Oil_Spill_mask.7z']:
    if not pathlib.Path(name).exists():
        print('downloading', name)
        subprocess.run(['curl','-L','--fail','-o',name,f'{B}/{name}/content'], check=True)
subprocess.run(['apt-get','-qq','install','-y','p7zip-full'], check=True)
for name in ['01_Train_Val_Oil_Spill_images.7z', '01_Train_Val_Oil_Spill_mask.7z']:
    subprocess.run(['7z','x','-y',name,'-o./data'], check=True)
print(sorted(os.listdir('data')))

In [ ]:
# --- U-Net (identical to neuro/backend/app/ml/unet.py) ---
import torch
from torch import nn

class _DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=32):
        super().__init__()
        c = [base, base*2, base*4, base*8]
        self.enc1=_DoubleConv(in_ch,c[0]); self.enc2=_DoubleConv(c[0],c[1]); self.enc3=_DoubleConv(c[1],c[2])
        self.pool=nn.MaxPool2d(2); self.bottleneck=_DoubleConv(c[2],c[3])
        self.up3=nn.ConvTranspose2d(c[3],c[2],2,stride=2); self.dec3=_DoubleConv(c[3],c[2])
        self.up2=nn.ConvTranspose2d(c[2],c[1],2,stride=2); self.dec2=_DoubleConv(c[2],c[1])
        self.up1=nn.ConvTranspose2d(c[1],c[0],2,stride=2); self.dec1=_DoubleConv(c[1],c[0])
        self.head=nn.Conv2d(c[0],out_ch,1)
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2))
        b=self.bottleneck(self.pool(e3))
        d3=self.dec3(torch.cat([self.up3(b),e3],1)); d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.head(d1)

def dice_bce_loss(logits, target, eps=1e-6):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    inter = (p*target).sum((1,2,3)); union = p.sum((1,2,3)) + target.sum((1,2,3))
    return bce + (1.0 - ((2*inter+eps)/(union+eps)).mean())

In [ ]:
# --- dataset (identical logic to neuro/backend/app/ml/dataset.py) ---
import random, numpy as np, tifffile
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
_DB_LO, _DB_HI = -35.0, 0.0

def normalise_db(a):
    a = np.clip(a.astype(np.float32), _DB_LO, _DB_HI)
    return (a - _DB_LO) / (_DB_HI - _DB_LO) * 2.0 - 1.0

def _list_pairs(root):
    masks, images = {}, {}
    for tif in Path(root).rglob('*.tif*'):
        (masks if 'mask' in tif.parent.name.lower() else images)[tif.stem] = tif
    return [(images[s], masks[s]) for s in sorted(images) if s in masks]

class OilSpillDataset(Dataset):
    def __init__(self, root, split='train', crop=512, val_fraction=0.1, seed=0):
        self.crop, self.split = crop, split
        pairs = _list_pairs(root)
        assert pairs, f'no image/mask pairs under {root}'
        random.Random(seed).shuffle(pairs)
        cut = max(1, int(len(pairs)*val_fraction))
        self.pairs = pairs[cut:] if split == 'train' else pairs[:cut]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        img = tifffile.imread(ip); msk = tifffile.imread(mp)
        if img.ndim == 2: img = np.stack([img, img], -1)
        img = normalise_db(img); msk = (msk > 0).astype(np.float32)
        h, w = msk.shape; c = min(self.crop, h, w)
        if self.split == 'train':
            y, x = random.randint(0, h-c), random.randint(0, w-c)
        else:
            y, x = (h-c)//2, (w-c)//2
        img, msk = img[y:y+c, x:x+c, :], msk[y:y+c, x:x+c]
        if self.split == 'train':
            if random.random() < 0.5: img, msk = img[:, ::-1, :], msk[:, ::-1]
            if random.random() < 0.5: img, msk = img[::-1, :, :], msk[::-1, :]
        it = torch.from_numpy(np.ascontiguousarray(img.transpose(2,0,1))).float()
        mt = torch.from_numpy(np.ascontiguousarray(msk))[None].float()
        return it, mt

In [ ]:
# --- train ---
import time
DATA='data'; EPOCHS=50; BATCH=12; CROP=512; BASE=32; LR=1e-3; OUT='oilspill_unet.pt'
dev = torch.device('cuda'); torch.manual_seed(0)
tr = OilSpillDataset(DATA, 'train', CROP); va = OilSpillDataset(DATA, 'val', CROP)
print(f'train {len(tr)} / val {len(va)}')
tl = DataLoader(tr, BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
vl = DataLoader(va, BATCH, shuffle=False, num_workers=2)
model = UNet(2, 1, BASE).to(dev); opt = torch.optim.Adam(model.parameters(), LR)

@torch.no_grad()
def iou(logits, t, thr=0.5):
    p = (torch.sigmoid(logits) > thr).float()
    i = (p*t).sum().item(); u = (p + t - p*t).sum().item()
    return i/u if u > 0 else 1.0

best = -1
for ep in range(1, EPOCHS+1):
    model.train(); t0 = time.time(); run = 0.0
    for img, msk in tl:
        img, msk = img.to(dev), msk.to(dev)
        opt.zero_grad(); loss = dice_bce_loss(model(img), msk); loss.backward(); opt.step()
        run += loss.item()
    model.eval(); ious = []
    with torch.no_grad():
        for img, msk in vl: ious.append(iou(model(img.to(dev)), msk.to(dev)))
    v = sum(ious)/max(len(ious),1)
    print(f'epoch {ep:3d}  loss {run/max(len(tl),1):.4f}  val_iou {v:.3f}  {time.time()-t0:.0f}s')
    if v > best:
        best = v
        torch.save({'state_dict': model.state_dict(), 'in_ch': 2, 'base': BASE,
                    'crop': CROP, 'val_iou': round(v,4),
                    'meta': {'dataset': 'zenodo-8346860', 'epochs_run': ep}}, OUT)
        print('  saved', OUT, f'(val_iou {v:.3f})')

In [ ]:
from google.colab import files
files.download('oilspill_unet.pt')

## Then, locally
```
mkdir -p neuro/backend/weights
mv ~/Downloads/oilspill_unet.pt neuro/backend/weights/
```
`neuro/backend/.env`:
```
SEGMENTATION_MODEL=trained-unet
SEGMENTATION_WEIGHTS=weights/oilspill_unet.pt
```
Restart the backend. `TrainedUNetSegmentationModel` picks it up; detection runs on the model,
falling back to the scenario slick only if the map comes back empty.